## RAG 1. Install dependencies

In [ ]:
!pip install llama-index llama-index-llms-openai llama-index-embeddings-openai openai

## RAG 2. Configuration

In [ ]:
import os

# Path to your locally cloned NEURON repo's docs folder
DOCS_PATHS = [
    ".."
    "docs/nmodl",     # <-- replace with your local path
    "docs/progref",  # <-- replace with your local path
]
  # <-- replace with your local path

# Where to save the persistent index (so you don't re-index every time)
INDEX_STORE_PATH = "docs/neuron_index"

## RAG 3. Build (or load) the index

In [ ]:
from llama_index.core import (
    VectorStoreIndex,
    SimpleDirectoryReader,
    StorageContext,
    load_index_from_storage,
)
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core import Settings

# Configure models
Settings.llm = OpenAI(model="gpt-4o", temperature=0.2)
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")

if os.path.exists(INDEX_STORE_PATH):
    print("Loading existing index...")
    storage_context = StorageContext.from_defaults(persist_dir=INDEX_STORE_PATH)
    index = load_index_from_storage(storage_context)
else:
    print("Building index from docs (this may take a few minutes)...")
    all_documents = []
    for docs_path in DOCS_PATHS:
        docs = SimpleDirectoryReader(
            input_dir=docs_path,
            recursive=True,
            required_exts=[".rst"],
        ).load_data()
        print(f"Loaded {len(docs)} .rst files from {docs_path}")
        all_documents.extend(docs)
    print(f"Total: {len(all_documents)} .rst files")
    index = VectorStoreIndex.from_documents(all_documents)
    index.storage_context.persist(persist_dir=INDEX_STORE_PATH)
    print("Index built and saved.")

## RAG 4. Define the prompt template

In [ ]:
PROMPT_TEMPLATE = """
You are a technical documentation assistant helping integrate community
Q&A content into the NEURON simulator's official documentation.

Below are the most relevant excerpts from the current NEURON documentation,
each labelled with its source .rst file path:

---------------------
{context_str}
---------------------

Here is a forum Q&A thread that contains information to be integrated:

<thread>
{query_str}
</thread>

Instructions:
- Identify every function, method, or class that the thread adds new information about.
- For each one, produce a separate JSON object with exactly these fields:
    - "header": the function/method/class name (e.g. "NetCon.record", "CVode.event", "fadvance")
    - "rst_file": the .rst source file path from the context above (e.g. "docs/hoc/simctrl/cvode.rst")
    - "new_doc": the new .rst text to be inserted, written to match the style of the existing docs.
                 Include code examples where relevant. Be concise and technical.

Return ONLY a JSON array of these objects, with no preamble or explanation.
Example output format:
[
  {
    "header": "NetCon.record",
    "rst_file": "docs/hoc/modelspec/programmatic/network/netcon.rst",
    "new_doc": ".. note::\n\n   ``record()`` can accept a string ...",
    "flag": null
  },
  {
    "header": "CVode.event",
    "rst_file": "docs/hoc/simctrl/cvode.rst",
    "new_doc": "``CVode.event()`` can also accept a proc ...",
    "flag": null
  }
]

Only include information clearly supported by the thread. Do not extrapolate.
"""

## RAG 5. Run a query for a single forum thread

In [ ]:
import json
# Paste your forum thread here
with open("d_lst_posts.json", "r", encoding="utf-8") as f:
    entry = json.load(f)

In [ ]:
import json
from llama_index.core import PromptTemplate

query_engine = index.as_query_engine(
    similarity_top_k=5,
    text_qa_template=PromptTemplate(PROMPT_TEMPLATE),
)

def process_thread(thread_id, thread_text):
    """Query the index with a forum thread and return structured JSON results."""
    # Warn if thread is likely to be large (rough estimate: 1 token ~ 4 chars)
    estimated_tokens = len(thread_text) // 4
    if estimated_tokens > 8000:
        print(f"WARNING: Thread {thread_id} is large (~{estimated_tokens} tokens) and may hit the token limit.")
    try:
        response = query_engine.query(thread_text)
        raw = str(response).strip()
        # Strip markdown code fences if present
        if raw.startswith("```"):
            raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()
        try:
            return json.loads(raw)
        except json.JSONDecodeError:
            print(f"WARNING: {thread_id} — could not parse JSON response.")
            print("Raw response:")
            print(raw)
            return [{"header": "PARSE_ERROR", "rst_file": None, "new_doc": raw, "flag": "JSON parse failed"}]
    except Exception as e:
        error_msg = str(e)
        if "maximum context length" in error_msg or "token" in error_msg.lower():
            print(f"ERROR: {thread_id} exceeded the token limit. (~{estimated_tokens} tokens, {len(thread_text)} chars)")
            print("Tip: truncate this thread or reduce similarity_top_k.")
        else:
            print(f"ERROR: {thread_id} failed with: {error_msg}")
        return [{"header": "TOKEN_ERROR", "rst_file": None, "new_doc": None, "flag": f"Failed: {error_msg}"}]


# entry is a list of dicts, so select one entry (e.g., the first)
forum_id = entry[0]["id"]
forum_thread = entry[0]["post"]

result = process_thread(forum_id, forum_thread)
print(json.dumps(result, indent=2))

## RAG 6. Bulk processing

In [ ]:
# Collect all entries from all threads into a single flat list
all_results = []
for item in entry:
    thread_id = item['id']
    thread_text = item['post']
    print(f"Processing thread {thread_id}...")
    entries = process_thread(thread_id, thread_text)
    for e in entries:
        e["source_thread"] = thread_id  # track which thread it came from
    all_results.extend(entries)

print(f"\nTotal entries: {len(all_results)}")
print(json.dumps(all_results, indent=2))

## RAG 7. Save results to a file for review

In [ ]:
# output_file = f"docs/neuron_doc_suggestions.json"

# with open(output_file, "w") as f:
#     json.dump(all_results, f, indent=2)

# print(f"Saved {len(all_results)} entries to {output_file}")